##  2 – Tự viết K-Means

In [1]:
# -*- coding: utf-8 -*-


import numpy as np
from sklearn.cluster import KMeans

P = np.array([[1.0, 1.0], [1.5, 2.0], [3.0, 4.0], [5.0, 7.0],
             [3.5, 5.0], [4.5, 5.0], [3.5, 4.5]])
TEN = ["A", "B", "C", "D", "E", "F", "G"]


def k_means(X, tam, so_vong_toi_da=10):
    """K-Means don gian: gan diem vao tam gan nhat roi doi tam."""
    for vong in range(1, so_vong_toi_da + 1):
       # Bước gán: khoảng cách từmỗi điểm tới mỗi tâm
       kc = np.sqrt(((X[:, None, :] - tam[None, :, :]) ** 2).sum(axis=2))
       nhan = kc.argmin(axis=1)
       print(f"Vong {vong}: tam = {np.round(tam, 2).tolist()}")
       for k in range(len(tam)):
           ds = [TEN[i] for i in range(len(X)) if nhan[i] == k]
           print(f"cum {k + 1}: {', '.join(ds)}")
       # Bước dời tâm: trung bình các điểm trong mỗi cụm
       tam_moi = np.array([X[nhan == k].mean(axis=0)
                          for k in range(len(tam))])
       if np.allclose(tam_moi, tam):
           print("Tam khong doi nua, dung.")
           return tam, nhan
       tam = tam_moi
    return tam, nhan


# Khởi tạo cốý tệ: hai tâm là A và B, nằm sát nhau
tam, nhan = k_means(P, P[[0, 1]].copy())
print()

# Tính inertia: tổng bình phương khoảng cách tới tâm cụm của mình
inertia = ((P - tam[nhan]) ** 2).sum()
print(f"Inertia tu tinh: {inertia:.4f}")

# So với scikit-learn, cho cùng tâm ban đầu
km = KMeans(n_clusters=2, init=P[[0, 1]], n_init=1).fit(P)
print(f"Inertia cua scikit-learn: {km.inertia_:.4f}")
print("Tam cua scikit-learn:", np.round(km.cluster_centers_, 2).tolist())


Vong 1: tam = [[1.0, 1.0], [1.5, 2.0]]
cum 1: A
cum 2: B, C, D, E, F, G
Vong 2: tam = [[1.0, 1.0], [3.5, 4.58]]
cum 1: A, B
cum 2: C, D, E, F, G
Vong 3: tam = [[1.25, 1.5], [3.9, 5.1]]
cum 1: A, B
cum 2: C, D, E, F, G
Tam khong doi nua, dung.

Inertia tu tinh: 8.5250
Inertia cua scikit-learn: 8.5250
Tam cua scikit-learn: [[1.25, 1.5], [3.9, 5.1]]


##  3 – Phân khúc khách hàng

In [2]:
# -*- coding: utf-8 -*-
"""Bước 2: phân khúc khách hàng bằng K-Means."""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/khach_hang.csv")
print("Kich thuoc bang:", df.shape)
print(df.describe().round(1).to_string())
print()

X = df[["thu_nhap", "diem_chi_tieu"]]
sc = StandardScaler().fit(X)
Xs = sc.transform(X)

km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
df["cum"] = km.labels_

# Tâm cụm đang ởthang đã chuẩn hoá, đổi ngược vềđơn vịgốc đểđọc
tam = pd.DataFrame(sc.inverse_transform(km.cluster_centers_),
                 columns=["thu_nhap", "diem_chi_tieu"])
tam["so_khach"] = np.bincount(km.labels_)
tam["tuoi_tb"] = df.groupby("cum")["tuoi"].mean()
print("Tam cum, don vi goc:")
print(tam.round(1).to_string())
print()
print(f"Inertia: {km.inertia_:.2f}")
print()

# Khách mới: chuẩn hoá bằng đúng bộchuẩn hoá cũ rồi tìm tâm gần nhất
moi = pd.DataFrame({"thu_nhap": [29.0, 9.0],
                  "diem_chi_tieu": [85, 75]})
cum_moi = km.predict(sc.transform(moi))
for i in range(len(moi)):
    print(f"Khach moi {i + 1}: thu nhap {moi.iloc[i, 0]},"
         f" chi tieu {moi.iloc[i, 1]} -> cum {cum_moi[i]}")


Kich thuoc bang: (210, 3)
        tuoi  thu_nhap  diem_chi_tieu
count  210.0     210.0          210.0
mean    37.6      18.5           49.9
std     10.3       9.8           26.8
min     18.0       3.0            7.0
25%     30.0       8.8           22.0
50%     39.0      17.4           49.0
75%     46.0      28.6           77.0
max     62.0      35.5           99.0

Tam cum, don vi goc:
   thu_nhap  diem_chi_tieu  so_khach  tuoi_tb
0       7.5           80.2        35     24.9
1      30.1           20.0        35     42.9
2      17.4           49.3        60     40.4
3       8.0           18.9        40     44.9
4      30.2           81.5        40     32.6

Inertia: 23.02

Khach moi 1: thu nhap 29.0, chi tieu 85 -> cum 4
Khach moi 2: thu nhap 9.0, chi tieu 75 -> cum 0


##  4 – Chọn số cụm K

In [3]:
# -*- coding: utf-8 -*-
"""Bước 3: chọn sốcụm K bằng khuỷu tay và chỉsốsilhouette."""

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/khach_hang.csv")
Xs = StandardScaler().fit_transform(df[["thu_nhap", "diem_chi_tieu"]])

print("Kinertiagiam so voi K-1silhouette")
truoc = None
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs)
    giam = "" if truoc is None else f"{truoc - km.inertia_:8.1f}"
    # Silhouette cần ít nhất hai cụm
    sil = "" if k == 1 else f"{silhouette_score(Xs, km.labels_):.3f}"
    print(f"{k:2d}{km.inertia_:7.1f}{giam:>15s}{sil:>10s}")
    truoc = km.inertia_


Kinertiagiam so voi K-1silhouette
 1  420.0                         
 2  250.4          169.6     0.430
 3  152.5           97.9     0.522
 4   73.8           78.8     0.626
 5   23.0           50.7     0.723
 6   20.9            2.1     0.651
 7   19.0            1.9     0.541
 8   17.4            1.6     0.454
 9   15.9            1.5     0.471
10   14.4            1.4     0.425


##  5 – Ba cái bẫy của K-Means

In [4]:
# -*- coding: utf-8 -*-
"""Bước 4: ba cái bẫy của K-Means."""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/khach_hang.csv")
X = df[["thu_nhap", "diem_chi_tieu"]]
Xs = StandardScaler().fit_transform(X)

# Bẫy 1: thu nhập ghi bằng nghìn đồng, không chuẩn hoá
X_nghin = X.copy()
X_nghin["thu_nhap"] = X_nghin["thu_nhap"] * 1000
km_tho = KMeans(n_clusters=5, n_init=10, random_state=0).fit(X_nghin)
tam_tho = km_tho.cluster_centers_.copy()
tam_tho[:, 0] = tam_tho[:, 0] / 1000# đổi lại triệu đồng đểdễđọc
print("Bay 1: khong chuan hoa, tam cum (thu nhap trieu, chi tieu):")
for t in tam_tho[np.argsort(tam_tho[:, 0])]:
    print(f"({t[0]:5.1f}, {t[1]:5.1f})")
print()

# Bẫy 2: mỗi lần chỉkhởi tạo ngẫu nhiên một lần
ket = []
for rs in range(20):
    km = KMeans(n_clusters=5, init="random", n_init=1,
               random_state=rs).fit(Xs)
    ket.append(round(km.inertia_, 1))
print("Bay 2: inertia cua 20 lan khoi tao ngau nhien:")
ket = sorted(ket)
print(" ", ket[:10])
print(" ", ket[10:])
tot = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
print(f"Voi n_init=10: inertia {tot.inertia_:.1f}")
print()

# Bẫy 3: cụm không tròn, dữliệu hai vầng trăng
tr = pd.read_csv("data/hai_vang_trang.csv")
km_tr = KMeans(n_clusters=2, n_init=10, random_state=0)
nhan = km_tr.fit_predict(tr[["x1", "x2"]])
khop = (nhan == tr["nhan"]).mean()
# Sốthứtựcụm có thểngược với nhãn thật, nên lấy cách khớp tốt hơn
print(f"Bay 3: hai vang trang, ty le khop nhan that "
      f"{max(khop, 1 - khop):.4f}")


Bay 1: khong chuan hoa, tam cum (thu nhap trieu, chi tieu):
(  6.1,  52.5)
(  9.6,  41.9)
( 17.4,  49.8)
( 27.8,  52.9)
( 31.9,  52.4)

Bay 2: inertia cua 20 lan khoi tao ngau nhien:
  [23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0]
  [72.4, 73.6, 73.7, 73.7, 84.0, 84.2, 84.2, 116.6, 117.4, 118.2]
Voi n_init=10: inertia 23.0

Bay 3: hai vang trang, ty le khop nhan that 0.7500


##  6 – Nén màu ảnh

In [5]:
# -*- coding: utf-8 -*-
"""Bước 5: nén màu một bức ảnh bằng K-Means."""

import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image

anh = load_sample_image("china.jpg")# ảnh có sẵn trong scikit-learn
cao, rong, _ = anh.shape
diem = anh.reshape(-1, 3) / 255.0# mỗi dòng là một điểm ảnh RGB
so_mau = len(np.unique(anh.reshape(-1, 3), axis=0))
print(f"Anh {cao} x {rong}, co {cao * rong} diem anh, {so_mau} mau")
print()

# Học trên 5000 điểm ảnh rút ngẫu nhiên cho nhanh, rồi gán cho mọi điểm
rng = np.random.default_rng(0)
mau = diem[rng.choice(len(diem), 5000, replace=False)]

print("Kso bit moi diemnho honsai lech trung binh")
for k in [4, 16, 64]:
    km = KMeans(n_clusters=k, n_init=4, random_state=0).fit(mau)
    nhan = km.predict(diem)
    moi = km.cluster_centers_[nhan]
    bit = int(np.ceil(np.log2(k)))
    sai = np.abs(moi - diem).mean() * 255
    print(f"{k:2d}{bit}{24 / bit:4.1f} lan"
         f"{sai:5.2f} muc sang")


Anh 427 x 640, co 273280 diem anh, 96615 mau

Kso bit moi diemnho honsai lech trung binh
 4212.0 lan17.41 muc sang
164 6.0 lan 7.97 muc sang
646 4.0 lan 4.54 muc sang


## Bài tập

### Bài tập 1 – Tự tính silhouette

In [6]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

def silhouette_mot_diem(X, nhan, i):
    # a: khoảng cách trung bình từ i tới các điểm khác cùng cụm
    # b: khoảng cách trung bình từ i tới cụm khác gần nhất
    X = np.asarray(X)
    nhan = np.asarray(nhan)
    d = np.linalg.norm(X - X[i], axis=1)
    cung = np.where(nhan == nhan[i])[0]
    cung = cung[cung != i]
    if len(cung) == 0:
        return 0.0, 0.0, 0.0
    a = d[cung].mean()
    b = min(d[nhan == k].mean() for k in np.unique(nhan) if k != nhan[i])
    s = (b - a) / max(a, b) if max(a, b) else 0.0
    return a, b, s

df = pd.read_csv('data/khach_hang.csv')
X = StandardScaler().fit_transform(df[['thu_nhap', 'diem_chi_tieu']])
nhan = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(X)
s = np.array([silhouette_mot_diem(X, nhan, i)[2] for i in range(len(X))])
print('Silhouette tự tính:', round(s.mean(), 5))
print('Silhouette từ thư viện:', round(silhouette_score(X, nhan), 5))
print('Khách có silhouette thấp nhất - dòng:', int(s.argmin()), ', giá trị:', round(s.min(), 4))
print(df.iloc[s.argmin()].to_string())
assert np.allclose(s, silhouette_samples(X, nhan))


Silhouette tự tính: 0.72271
Silhouette từ thư viện: 0.72271
Khách có silhouette thấp nhất - dòng: 113 , giá trị: 0.0286
tuoi             38.0
thu_nhap         25.4
diem_chi_tieu    41.0


**Nhận xét**

- Silhouette càng gần 1 thì điểm nằm gọn trong cụm. Giá trị thấp thường là điểm ở gần ranh giới của hai cụm.

### Bài tập 2 – Thêm cột tuổi

In [7]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

df = pd.read_csv('data/khach_hang.csv')
cols2 = ['thu_nhap', 'diem_chi_tieu']
cols3 = ['tuoi', 'thu_nhap', 'diem_chi_tieu']
sc = StandardScaler().fit(df[cols3])
X = sc.transform(df[cols3])
diem = {}
for k in range(2, 11):
    nhan = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    diem[k] = silhouette_score(X, nhan)
    print('K =', k, '| silhouette =', round(diem[k], 4))
k_tot = max(diem, key=diem.get)
km3 = KMeans(n_clusters=k_tot, n_init=10, random_state=0).fit(X)
print('K tốt nhất khi có tuổi:', k_tot)
print('Tâm cụm 3 cột ở đơn vị gốc:')
print(pd.DataFrame(sc.inverse_transform(km3.cluster_centers_), columns=cols3).round(2).to_string(index=False))
sc2 = StandardScaler().fit(df[cols2])
km2 = KMeans(n_clusters=5, n_init=10, random_state=0).fit(sc2.transform(df[cols2]))
print('Tâm cụm 2 cột ban đầu:')
print(pd.DataFrame(sc2.inverse_transform(km2.cluster_centers_), columns=cols2).round(2).to_string(index=False))


K = 2 | silhouette = 0.373
K = 3 | silhouette = 0.422
K = 4 | silhouette = 0.4558
K = 5 | silhouette = 0.4808
K = 6 | silhouette = 0.476
K = 7 | silhouette = 0.4627
K = 8 | silhouette = 0.4596
K = 9 | silhouette = 0.4622
K = 10 | silhouette = 0.4573
K tốt nhất khi có tuổi: 5
Tâm cụm 3 cột ở đơn vị gốc:
 tuoi  thu_nhap  diem_chi_tieu
44.88      8.04          18.85
25.13      8.48          77.56
42.86     30.13          19.97
32.08     30.28          82.00
41.65     17.58          49.18
Tâm cụm 2 cột ban đầu:
 thu_nhap  diem_chi_tieu
     7.48          80.20
    30.13          19.97
    17.39          49.30
     8.04          18.85
    30.21          81.53


**Nhận xét**

- Thêm tuổi làm thay đổi cách tính khoảng cách giữa khách hàng. Vì vậy K tốt nhất và tâm cụm có thể đổi.

### Bài tập 3 – K-Means++ và khởi tạo ngẫu nhiên

In [8]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data/khach_hang.csv')
X = StandardScaler().fit_transform(df[['thu_nhap', 'diem_chi_tieu']])
for cach in ['random', 'k-means++']:
    inertia = []
    for seed in range(50):
        km = KMeans(n_clusters=5, init=cach, n_init=1, random_state=seed).fit(X)
        inertia.append(km.inertia_)
    so_lan = sum(round(v, 1) == 23.0 for v in inertia)
    print(cach, '| số lần inertia làm tròn bằng 23.0:', so_lan, '/ 50')
    print('  inertia nhỏ nhất:', round(min(inertia), 4), 'lớn nhất:', round(max(inertia), 4))


random | số lần inertia làm tròn bằng 23.0: 30 / 50
  inertia nhỏ nhất: 23.0167 lớn nhất: 204.0456
k-means++ | số lần inertia làm tròn bằng 23.0: 50 / 50
  inertia nhỏ nhất: 23.0167 lớn nhất: 23.0167


**Nhận xét**

- K-Means++ thường ổn định hơn vì chọn tâm ban đầu cách xa nhau.
- Em đếm số lần đạt mức inertia khoảng 23.0 thay vì chỉ nhìn lần chạy tốt nhất.

### Bài tập 4 – Lưu ảnh nén 16 màu

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image

def nen_16_mau(anh):
    anh = np.asarray(anh)[..., :3]
    anh = anh.astype(float)
    if anh.max() > 1:
        anh /= 255.0
    cao, rong, _ = anh.shape
    diem = anh.reshape(-1, 3)
    rng = np.random.default_rng(42)
    mau = diem[rng.choice(len(diem), min(5000, len(diem)), replace=False)]
    km = KMeans(n_clusters=16, n_init=4, random_state=0).fit(mau)
    anh_moi = km.cluster_centers_[km.predict(diem)].reshape(cao, rong, 3)
    return np.clip(anh_moi, 0, 1)

anh = load_sample_image('china.jpg')
anh_moi = nen_16_mau(anh)
plt.imsave('anh_goc.png', anh)
plt.imsave('anh_16_mau.png', anh_moi)
print('Kích thước ảnh nén:', anh_moi.shape)
for file in ['anh_goc.png', 'anh_16_mau.png']:
    print(file, Path(file).stat().st_size, 'bytes')
# Có ảnh cá nhân thì đặt cùng thư mục với notebook, tên anh_cua_em.png
file_ca_nhan = Path('anh_cua_em.png')
if file_ca_nhan.exists():
    anh_khac = plt.imread(file_ca_nhan)
    plt.imsave('anh_cua_em_16_mau.png', nen_16_mau(anh_khac))
    print('Đã lưu anh_cua_em_16_mau.png')
else:
    print('Chưa có anh_cua_em.png nên bỏ qua phần ảnh cá nhân.')


Kích thước ảnh nén: (427, 640, 3)
anh_goc.png 550154 bytes
anh_16_mau.png 134073 bytes
Chưa có anh_cua_em.png nên bỏ qua phần ảnh cá nhân.


**Nhận xét**

- Ảnh sau khi nén chỉ dùng 16 màu nên mất bớt chi tiết.
- Dung lượng PNG thực tế còn phụ thuộc cách lưu ảnh, không chỉ số lượng màu.

### Bài tập 5 – DBSCAN trên hai vầng trăng

In [10]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score

df = pd.read_csv('data/hai_vang_trang.csv')
X = df[['x1', 'x2']].to_numpy()
y = df['nhan'].to_numpy()
for eps in [0.2, 0.3, 0.4]:
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X)
    n_cum = len(set(labels) - {-1})
    n_nhieu = int((labels == -1).sum())
    print('eps =', eps, '| số cụm:', n_cum, '| điểm nhiễu:', n_nhieu,
          '| ARI:', round(adjusted_rand_score(y, labels), 4))
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)
print('K-Means ARI:', round(adjusted_rand_score(y, km), 4))


eps = 0.2 | số cụm: 1 | điểm nhiễu: 40 | ARI: -0.0014
eps = 0.3 | số cụm: 1 | điểm nhiễu: 10 | ARI: 0.0012
eps = 0.4 | số cụm: 1 | điểm nhiễu: 3 | ARI: -0.0001
K-Means ARI: 0.2475


**Nhận xét**

- Eps quá nhỏ dễ làm nhiều điểm thành nhiễu; eps quá lớn có thể gộp hai cụm.
- ARI càng gần 1 thì kết quả phân cụm càng giống nhãn thật.

### Bài tập 6 – Phân cụm chữ số viết tay

In [11]:
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X = digits.data
nhan_that = digits.target
# Không đưa nhãn thật vào quá trình phân cụm
cum = KMeans(n_clusters=10, n_init=10, random_state=0).fit_predict(X)
bang = pd.crosstab(pd.Series(nhan_that, name='Chữ số thật'), pd.Series(cum, name='Cụm'))
print(bang)
print('Cụm chứa nhiều ảnh nhất theo từng chữ số:')
print(bang.idxmax(axis=1).to_string())
print('Tỉ lệ tập trung cao nhất theo chữ số:')
print((bang.max(axis=1) / bang.sum(axis=1)).round(3).to_string())


Cụm            0    1    2   3    4    5    6    7    8    9
Chữ số thật                                                 
0            176    0    0   0    0    0    2    0    0    0
1              0  100    0  54    1    2    0    1    0   24
2              1    8    3   2   13    0    0    0    2  148
3              0    7    7   0  154    0    0    2   13    0
4              0    2   11   3    0    0  165    0    0    0
5              0    0    0   0    2    1    2  136   41    0
6              1    3    0   0    0  177    0    0    0    0
7              0    2  174   2    0    0    0    1    0    0
8              0  100    5   6    2    2    0    4   52    3
9              0    1    8  20    6    0    0    6  139    0
Cụm chứa nhiều ảnh nhất theo từng chữ số:
Chữ số thật
0    0
1    1
2    9
3    4
4    6
5    7
6    5
7    2
8    1
9    8
Tỉ lệ tập trung cao nhất theo chữ số:
Chữ số thật
0    0.989
1    0.549
2    0.836
3    0.842
4    0.912
5    0.747
6    0.978
7    0.972
8    0

**Nhận xét**

- K-Means không biết tên chữ số, chỉ nhóm những ảnh giống nhau.
- Một vài chữ số có nét gần giống nhau nên dễ bị gom chung cụm.